# nano Beatrix — the wide-band screen: Alhena's key-to-record lookup on a wider content band

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/geolip-nano-beatrix-trainer/blob/main/colab/nano_wide.ipynb)

**The question**: Alhena (Castor's body turned by the anchored multi-position rotary) answers a question exactly when it is handed the address it asks for, and dozens of independently trained arms then merge under the gate with nothing lost. The address has to come from the model itself: at each entry of the index it writes, the model must find the record that holds the entry's key. That lookup is unreliable: on some runs it locks in (99.5-100%), on others it stalls at 85-92%. A read of the attention scores puts the lookup in one head, matching the key to its record through the head's content pairs (unrotated) and its slowest position pairs; where the match is weak, a random other record wins. Alhena gives content only 2 of the head's 16 rotary pairs.

**Alhena-W** moves the split: 4 position pairs (NTK-aware x4 over their 8 dims; wavelengths 6.3 / 100 / 1,583 / 25,100 positions), 4 content pairs, and the same 8 address pairs (two digit levels, w_k = 2 pi k / 9, k = 1..4). Nothing else changes.

| part | what | why |
|---|---|---|
| W s1-s4 | Alhena-W, the scale test's recipe at 24,000 steps (CE + the address head at the questions) | does the model's own lookup (its written index) lock on every seed |
| A s3, s4 | Alhena verbatim on two new seeds (series `scalerep`) | the replication arm: the base lock rate (seed 1's two runs and seed 2 so far: 1 of 3) |
| WF s1, s2 | Alhena-W with the head also trained at the index entries (weight 1) | the find form on the wide band (on Alhena it stalled on both seeds at 24,000 steps) |
| members | the scale test's 48 real arms + 48 toy members (after the last block) and 16 real arms (after block 3), each trained alone, on W s1 and W s2 | the request = the model's own written index (free-running); merged under w_i = sinh(u_i) / (1 + sum_j (cosh(u_j) - 1)) |

**The cells**: C1 pins the trainer and the arm library and holds the decision block; C2 builds the data, runs 200 steps of Alhena-W on this card for the price line (a projection past 1.5x the plan stops for a re-plan) and probes the push path; C3 trains W s1 and W s2; C4 trains the members on each of them whose written index is right on at least .98 of its entries (free-running) and whose answers with the true address reach .95; C5 trains the other cores; C6 trains the members on a seed's WF core where its W core did not qualify; C7 renders the tables. Every finished run ships to [AbstractPhil/alephllm-nano-beatrix-training](https://huggingface.co/AbstractPhil/alephllm-nano-beatrix-training) (`bed/reports/`, `bed/weights/<run>/`, `bed/scale_members/`).

**Before C1**: the notebook's `HF_TOKEN` secret (write access to that repo) with notebook access on. Run C1, then Runtime > Restart session and run all. A file named `STOP` in `/content/out` ends the session between runs; a restarted session resumes where it stopped. The code and its tests: [AbstractEyes/geolip-nano-beatrix-trainer](https://github.com/AbstractEyes/geolip-nano-beatrix-trainer) (`geolip/nano_beatrix/anchored_rope.py`, `rotary_scale.py`).

In [ ]:
# C1 — pinned installs + THE DECISION BLOCK. RESTART the session after the first run of this cell (Runtime > Restart session and run all).
import os, sys
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'   # a progress bar is not allowed to cost a session
%pip install -q "geolip-nano-beatrix-trainer[corpus] @ git+https://github.com/AbstractEyes/geolip-nano-beatrix-trainer@v0.1.73" "amoe-lora @ git+https://github.com/AbstractEyes/amoe-lora@v0.2.12"
import torch, amoe
import geolip.nano_beatrix as N
print('geolip.nano_beatrix', N.__version__, '| amoe', amoe.__version__, '| torch', torch.__version__, '| card',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert tuple(int(x) for x in N.__version__.split('.')) >= (0, 1, 73), (
    'RESTART REQUIRED: the runtime still holds an older package — Runtime > Restart session, then rerun from C1')
assert amoe.__version__ == '0.2.12', 'the arm module of record is amoe-lora 0.2.12'
assert torch.cuda.is_available(), 'no card: Runtime > Change runtime type > GPU'

def _token():
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception:
        return os.environ.get('HF_TOKEN')
HF_TOKEN = _token()
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN            # the runner and the shipper read it from the environment; the value is never printed

# ============ THE DECISION BLOCK — every setting set from the plan, with its source beside it ============
OUT = '/content/out'                                   # the session folder: the corpus, the records, the weights, the members, runner.log
REPO = 'AbstractPhil/alephllm-nano-beatrix-training'  # every finished run ships here; tokenless = nothing ships
CORPUS_BYTES = 160e6                                   # fineweb-edu text for the bed (the scale test's corpus)
BED = dict(                                            # the scale test's Alhena recipe at 24,000 steps, verbatim
    ctx=4096, mb=8, steps=24000,                       # 24,000 steps x 8 rows x 4,096 bytes; pure Adam, lr 1e-3, 100 warm-up steps, the cosine to a tenth
    frames='dyn', plant=1.0,
    sizes='32,128,256,512', frame_records=4, nest=0.3, overlap=0.0, ask='any', register='caption', tags='mode',
    log_every=100,
)
GRID_ROWS = dict(grid='ag', grid_p=0.5,                # half of every batch's rows are addressed grid rows
                 head='digit', head_lam=1.0,           # the per-digit address head at the questions' SEPs (the scale test's term)
                 trace_every=2000)                     # the formation trace every 2,000 steps
WIDE = dict(rotary_content=4, series='wide')           # ALHENA-W: 4 relative / 4 content / 8 anchor pairs (the run's name carries _cp4)
FIND = dict(head_at='both', head_lam_i=1.0)            # the find screen's term: the same head at each index entry's SEP, weight 1
W = {s: (f'W s{s}', dict(BED, **GRID_ROWS, **WIDE, arms='alhena-both', seeds=str(s))) for s in (1, 2, 3, 4)}
A = {s: (f'A s{s}', dict(BED, **GRID_ROWS, arms='alhena-both', seeds=str(s), series='scalerep')) for s in (3, 4)}   # the replication arm
WF = {s: (f'WF s{s}', dict(BED, **GRID_ROWS, **WIDE, **FIND, arms='alhena-both', seeds=str(s))) for s in (1, 2)}
GATE = [W[1], W[2]]                                    # C3: they gate the members
REST = [A[3], W[3], A[4], W[4], WF[1], WF[2]]          # C5: matched pairs first, so a stop leaves pairs whole
TAIL = '_L24000_ctx4096_frdyn_cap_mode_sz32-128-256-512_r4_n30_o0_askany_gag50'
TAG_W = lambda s: f'alhena-both_s{s}{TAIL}_cp4_ahdigit1_wide'
TAG_WF = lambda s: f'alhena-both_s{s}{TAIL}_cp4_ahdigit1i1_wide'
MIN_INDEX, MIN_FIND, MIN_ANSWERS = 0.98, 0.98, 0.95    # the plan's member rule: index (free-running) or find >= .98, answers >= .95
LAST_STEPS = 16000                                     # the scale test's member budget, verbatim
MID_STEPS = 32000                                      # batch steps at the mid site (16,000 own steps a member)
READ_BATCHES, CALIB_BATCHES = 64, 32                   # the scale test's read and calibration rows (8 rows a batch)
SMOKE_STEPS = 200                                      # C2's short run of Alhena-W on this card for the price line; its record stays local
PRICE_STEPS = 200                                      # C4's member price lines (each trainer on throwaway stacks)
PLAN_MIN = 16.5                                        # the planned minutes of one 24,000-step core on this card (981-1,000 s measured)
PLAN_MIN_MEM = 18.8                                    # the planned training minutes of one core's members (4.9 last + 13.9 mid)
STOP_FACTOR = 1.5                                      # a projection past 1.5x the plan stops the session for a re-plan
PARALLEL = 1                                           # ONE run at a time: one 9M craft at 8 x 4,096 saturates the card
print('decision block:', len(GATE) + len(REST), 'cores | the member rule', MIN_INDEX, MIN_ANSWERS, '| out', OUT, '| ships to',
      REPO if HF_TOKEN else 'nothing (tokenless)')

In [ ]:
# C2 (P) — PREFLIGHT: the data, the plan, 200 steps of Alhena-W on this card (the price line), the push probe. No part trains.
import json, os, time, torch
from geolip.nano_beatrix import bed_corpus, bed_runner
if not os.path.exists(f'{OUT}/bed/bed_corpus.bin'):
    bed_corpus.build(f'{OUT}/bed', CORPUS_BYTES)
HELD = f'{OUT}/bed/heldout_fineweb_edu_head2048.jsonl'
print('corpus', os.path.getsize(f'{OUT}/bed/bed_corpus.bin') / 1e6, 'MB; held-out head', os.path.exists(HELD))
todo = 0
for name, spec in GATE + REST:
    p = bed_runner.run(spec, out=OUT, dry=True)
    todo += p['runs'] - p['done_before']
smoke = dict(GATE[0][1], steps=SMOKE_STEPS, log_every=20, trace_every=100, read_batches=4, calib_batches=2)
res = bed_runner.run(smoke, out=OUT)
assert res['failed'] is None, 'the short run failed: read the lines above before any part'
last = [s for s in (bed_runner.parse_step(l) for l in open(f'{OUT}/runner.log', encoding='utf-8')) if s][-1]
per_run = BED['steps'] * BED['ctx'] * BED['mb'] / (last['tok_s'] * 1e3) / 60
PRICE_OK = per_run <= STOP_FACTOR * PLAN_MIN
print(f"this card: {last['tok_s']}k tok/s at {BED['ctx']} x {BED['mb']} -> one {BED['steps']:,}-step run about {per_run:.1f} min (planned "
      f"{PLAN_MIN} min); the {todo} runs left about {per_run * todo / 60:.1f} h (+ a few minutes of end reads a run, + the members)"
      + ('' if PRICE_OK else f' -> PAST {STOP_FACTOR}x THE PLAN: C3 will not start; re-plan first'))
PUSH_OK = False
if HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    path = 'bed/preflight/push_probe_wide.json'
    body = json.dumps({'utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'card': torch.cuda.get_device_name(0),
                       'tok_s': last['tok_s'], 'parts': [n for n, _ in GATE + REST]}).encode()
    api.upload_file(path_or_fileobj=body, path_in_repo=path, repo_id=REPO, commit_message='nano wide band: push probe')
    PUSH_OK = path in set(api.list_repo_files(REPO))
    print('push probe', 'READ BACK OK' if PUSH_OK else 'NOT VISIBLE after upload', path)
else:
    print('TOKENLESS: no probe written; the cells below run and ship nothing')

In [ ]:
# C3 (T) — W s1 and W s2 (they gate the members), one run at a time, resume-first (runner_done.txt in OUT); a STOP file in OUT starts
# no new run; every finished run ships to REPO as it lands (one background shipper: the card never waits on the Hub).
PARTS_NOW = GATE
from geolip.nano_beatrix import bed_runner
assert PRICE_OK, 'the price line projected past the plan: re-plan before any part trains'
push = REPO if (HF_TOKEN and PUSH_OK) else ''
shipper = globals().get('shipper') or (bed_runner._Shipper(push, OUT, print) if push else None)
if shipper:
    for rec in bed_runner.pending_records(OUT):
        shipper.submit(rec)
done = []
for name, spec in PARTS_NOW:
    if os.path.exists(f'{OUT}/STOP'):
        print('STOP present: no new part starts')
        break
    print(f'=================== {name} ===================')
    res = bed_runner.run(spec, out=OUT, push=push, parallel=PARALLEL, shipper=shipper)
    done.append((name, res))
    if res['failed'] is not None or res['stopped']:
        print('the part did not finish:', name, {k: v for k, v in res.items() if k != 'ran'})
        break
if shipper:
    shipper.drain(print)
    left = [os.path.basename(p)[4:-5] for p in bed_runner.pending_records(OUT)]
    print('UNSHIPPED:', left if left else 'none')
print('session:', [(n, r['failed'], r['stopped'], len(r['ran'])) for n, r in done])

In [ ]:
# C4 (T) — THE MEMBERS on W s1 and W s2: each W core whose own end read writes its index right on >= .98 of the entries (free-running) and
# answers >= .95 with the true address carries the scale test's members verbatim; the request is the core's own written index (the
# two-pass read, the question-head set: LAYOUT, INDEX, INDEXC, INDEXF, HEADPF, HEADPFQ999, HEADI, HEADIF, G2, G0). A W seed that does not
# qualify waits for its WF core (C6).
import json, time
from geolip.nano_beatrix import rotary_scale as rs
BED_DIR = f'{OUT}/bed'

def _record(tag):
    p = f'{BED_DIR}/bed_{tag}.json'
    return json.load(open(p, encoding='utf-8')) if os.path.exists(p) else None

def _qualifies(tag, find):
    r = _record(tag)
    if r is None:
        return None, 'no record in this session'
    g = r['grid']['read']
    fr = g.get('free_running') or {}
    ans = g['true']['whole']
    look = fr['headif']['right'] if find else fr.get('indexf_right', 0.0)
    ok = look >= (MIN_FIND if find else MIN_INDEX) and ans >= MIN_ANSWERS
    return ok, f"{'the find' if find else 'its written index (free-running)'} {look:.4f}, answers with the true address {ans:.4f}"

def run_members(cores):
    owned, never = rs.owned_sets()
    packed = rs.load_packed(BED_DIR)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    core = rs.Core(f'{BED_DIR}/{cores[0][0]}', 'cuda')
    cc = rs.compile_check(core, packed, owned, never)
    print('compiled = eager at step 1 (largest difference over the largest value):', cc)
    assert all(v['write'] < 1e-4 and v['grad'] < 1e-3 for v in cc.values()), 'the compiled writes are not the eager writes: no member trains'
    pr = rs.price(core, packed, owned, never, steps=PRICE_STEPS)
    per_core = (pr['last_ms_per_step'] * LAST_STEPS + pr['mid_ms_per_step'] * MID_STEPS) / 60000
    print(f"this card: last site {pr['last_ms_per_step']:.1f} ms a step, mid site {pr['mid_ms_per_step']:.1f} ms a step -> a core's members "
          f"about {per_core:.0f} min of training (planned {PLAN_MIN_MEM}) + the reads")
    assert per_core <= STOP_FACTOR * PLAN_MIN_MEM, 'the members priced past the plan: re-plan before any member trains'
    del core
    torch.cuda.empty_cache()
    ship = rs.Shipper(REPO) if (HF_TOKEN and PUSH_OK) else None
    t_all = time.time()
    out = []
    for n, (tag, seed) in enumerate(cores, 1):
        if os.path.exists(f'{OUT}/STOP'):
            print('STOP present: no new core starts')
            break
        print(f'=================== members {n}/{len(cores)} {tag} ===================')
        t0 = time.time()
        res = rs.run_core(f'{BED_DIR}/{tag}', f'{OUT}/members', seed, BED_DIR, HELD, last_steps=LAST_STEPS, mid_steps=MID_STEPS,
                          read_batches=READ_BATCHES, calib_batches=CALIB_BATCHES, shipper=ship, forms=('find',))
        out.append(res)
        torch.cuda.empty_cache()
        print(f'{tag}: {(time.time() - t0) / 60:.1f} min; {(time.time() - t_all) / 60:.1f} min spent; {len(cores) - n} cores left')
    if ship:
        print('ships drained' if ship.drain() else 'ships still retrying in the background')
    return out
MEMBER_CORES, WAIT = [], []
for s in (1, 2):
    ok, why = _qualifies(TAG_W(s), find=False)
    print(f'W s{s}: {why} -> ' + ('its members train' if ok else 'NOT HERE (C6 tries its WF core)'))
    (MEMBER_CORES if ok else WAIT).append((TAG_W(s), s) if ok else s)
done_m = run_members(MEMBER_CORES) if MEMBER_CORES else []
print('members:', [r['name'] for r in done_m], '| waiting for WF:', WAIT)

In [ ]:
# C5 (T) — THE OTHER CORES: A s3, W s3, A s4, W s4 (matched pairs), then WF s1, WF s2; one run at a time, resume-first, shipped as each lands.
PARTS_NOW = REST
from geolip.nano_beatrix import bed_runner
assert PRICE_OK, 'the price line projected past the plan: re-plan before any part trains'
push = REPO if (HF_TOKEN and PUSH_OK) else ''
shipper = globals().get('shipper') or (bed_runner._Shipper(push, OUT, print) if push else None)
if shipper:
    for rec in bed_runner.pending_records(OUT):
        shipper.submit(rec)
done = []
for name, spec in PARTS_NOW:
    if os.path.exists(f'{OUT}/STOP'):
        print('STOP present: no new part starts')
        break
    print(f'=================== {name} ===================')
    res = bed_runner.run(spec, out=OUT, push=push, parallel=PARALLEL, shipper=shipper)
    done.append((name, res))
    if res['failed'] is not None or res['stopped']:
        print('the part did not finish:', name, {k: v for k, v in res.items() if k != 'ran'})
        break
if shipper:
    shipper.drain(print)
    left = [os.path.basename(p)[4:-5] for p in bed_runner.pending_records(OUT)]
    print('UNSHIPPED:', left if left else 'none')
print('session:', [(n, r['failed'], r['stopped'], len(r['ran'])) for n, r in done])

In [ ]:
# C6 (T) — THE SUBSTITUTION RULE (the plan's, written before any run): a seed whose W core did not qualify in C4 carries its members on its WF
# core when that core's find (the head at the index entries, free-running) is right on >= .98 of the entries and its answers reach .95;
# at most two member cores in all. Nothing runs when both W cores qualified.
import json, time
from geolip.nano_beatrix import rotary_scale as rs
BED_DIR = f'{OUT}/bed'

def _record(tag):
    p = f'{BED_DIR}/bed_{tag}.json'
    return json.load(open(p, encoding='utf-8')) if os.path.exists(p) else None

def _qualifies(tag, find):
    r = _record(tag)
    if r is None:
        return None, 'no record in this session'
    g = r['grid']['read']
    fr = g.get('free_running') or {}
    ans = g['true']['whole']
    look = fr['headif']['right'] if find else fr.get('indexf_right', 0.0)
    ok = look >= (MIN_FIND if find else MIN_INDEX) and ans >= MIN_ANSWERS
    return ok, f"{'the find' if find else 'its written index (free-running)'} {look:.4f}, answers with the true address {ans:.4f}"

def run_members(cores):
    owned, never = rs.owned_sets()
    packed = rs.load_packed(BED_DIR)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    core = rs.Core(f'{BED_DIR}/{cores[0][0]}', 'cuda')
    cc = rs.compile_check(core, packed, owned, never)
    print('compiled = eager at step 1 (largest difference over the largest value):', cc)
    assert all(v['write'] < 1e-4 and v['grad'] < 1e-3 for v in cc.values()), 'the compiled writes are not the eager writes: no member trains'
    pr = rs.price(core, packed, owned, never, steps=PRICE_STEPS)
    per_core = (pr['last_ms_per_step'] * LAST_STEPS + pr['mid_ms_per_step'] * MID_STEPS) / 60000
    print(f"this card: last site {pr['last_ms_per_step']:.1f} ms a step, mid site {pr['mid_ms_per_step']:.1f} ms a step -> a core's members "
          f"about {per_core:.0f} min of training (planned {PLAN_MIN_MEM}) + the reads")
    assert per_core <= STOP_FACTOR * PLAN_MIN_MEM, 'the members priced past the plan: re-plan before any member trains'
    del core
    torch.cuda.empty_cache()
    ship = rs.Shipper(REPO) if (HF_TOKEN and PUSH_OK) else None
    t_all = time.time()
    out = []
    for n, (tag, seed) in enumerate(cores, 1):
        if os.path.exists(f'{OUT}/STOP'):
            print('STOP present: no new core starts')
            break
        print(f'=================== members {n}/{len(cores)} {tag} ===================')
        t0 = time.time()
        res = rs.run_core(f'{BED_DIR}/{tag}', f'{OUT}/members', seed, BED_DIR, HELD, last_steps=LAST_STEPS, mid_steps=MID_STEPS,
                          read_batches=READ_BATCHES, calib_batches=CALIB_BATCHES, shipper=ship, forms=('find',))
        out.append(res)
        torch.cuda.empty_cache()
        print(f'{tag}: {(time.time() - t0) / 60:.1f} min; {(time.time() - t_all) / 60:.1f} min spent; {len(cores) - n} cores left')
    if ship:
        print('ships drained' if ship.drain() else 'ships still retrying in the background')
    return out
SUB = []
for s in globals().get('WAIT', []):
    ok, why = _qualifies(TAG_WF(s), find=True)
    print(f'WF s{s}: {why} -> ' + ('its members train' if ok else 'NOT RUN (no member core for this seed)'))
    if ok:
        SUB.append((TAG_WF(s), s))
done_s = run_members(SUB) if SUB else []
print('substitute members:', [r['name'] for r in done_s] or 'none')

In [ ]:
# C7 — THE TABLES: the cores (bits, the frame copy, the index both ways, the find, the answers, minutes) and the members (per core and stack:
# solo against bare, the nine rest cells keeping all three marks per source, the controls, the largest leak of the written index, the mid
# site, the text bits). Run any time.
import glob, json
from IPython.display import Markdown, display
rows = ['| run | bits | copy 32 / 128 / 256 / 512 | index TF / FR | find right / truth / Q999 | answers true / via index / via find | min |',
        '|---|---|---|---|---|---|---|']
for p in sorted(glob.glob(f'{OUT}/bed/bed_alhena-both_*_L24000_*.json')):
    r = json.load(open(p, encoding='utf-8'))
    if not r.get('grid'):
        continue
    ff = r.get('fetch_by_frame') or {}
    copy = ' / '.join(f"{ff[f'data/{s}/depth0']['full_value']:.3f}" if f'data/{s}/depth0' in ff else '-' for s in (32, 128, 256, 512))
    g = r['grid']['read']
    fr = g.get('free_running') or {}
    hf = fr.get('headif')
    find = f"{hf['right']:.5f} / {hf['truth']:.4f} / {max(fr['calib']['tau']['0.999']):.2e}" if hf else '-'
    via = f"{g['true']['whole']:.4f} / {fr.get('indexf_whole', float('nan')):.4f} / " + (f"{hf['whole']:.4f}" if hf else '-')
    _n = os.path.basename(p)[4:-5]
    rows.append(f"| {_n.split('_ctx')[0]} {_n.split('gag50_')[-1]} | {r['heldout_bpb']:.4f} | {copy} | {g['true']['index_right']:.4f} / "
                f"{fr.get('indexf_right', float('nan')):.4f} | {find} | {via} | {r['train_minutes']:.1f} |")
display(Markdown('\n'.join(rows)))
rows = ['| core | stack | solo / bare owned | INDEXF | HEADPF | LAYOUT | G2 m48 | INDEXC m48 Q | G0 m48 P/Q | largest leak / own (INDEXF) | mid INDEXF U5/8/10 | G0 text bits |',
        '|---|---|---|---|---|---|---|---|---|---|---|---|']
for d in sorted(glob.glob(f'{OUT}/members/*_members_*')):
    if not os.path.exists(f'{d}/last_find.json'):
        continue
    L = json.load(open(f'{d}/last_find.json', encoding='utf-8'))
    M = json.load(open(f'{d}/mid_find.json', encoding='utf-8')) if os.path.exists(f'{d}/mid_find.json') else None
    rd = L['read']
    for st, r in rd['stacks'].items():
        c = r['cells']
        allm = lambda x: all(x[k] for k in ('PROTECTED', 'QUIET', 'EXACT'))
        nine = lambda s: f"{sum(allm(c[f'{s}_U{u}_REST_m{m}']) for u in (5, 8, 10) for m in (16, 32, 48))}/9" if f'{s}_U5_REST_m16' in c else '-'
        leak = max(c[f'INDEXF_U{u}_REST_m{m}']['leak_over_own_write'] for u in (5, 8, 10) for m in (16, 32, 48))
        mid = '-'
        if M and st == 'arms':
            mc = M['read']['cells']
            mid = ' '.join(str(allm(mc[f'INDEXF_U{u}_REST_m16'])) if f'INDEXF_U{u}_REST_m16' in mc else '-' for u in (5, 8, 10))
        g0 = L['text_bits'].get(f'G0_{st}_minus_bare')
        rows.append(f"| {L['core'][:30]} | {st} | {r['solo']['ONE']['48']:.3f} / {rd['bare']['owned']:.3f} | {nine('INDEXF')} | {nine('HEADPF')} | "
                    f"{nine('LAYOUT')} | {allm(c['G2_m48'])} | {c['INDEXC_U5_REST_m48']['QUIET']} | "
                    f"{c['G0_m48']['PROTECTED']}/{c['G0_m48']['QUIET']} | {leak:.1e} | {mid} | {g0:+.3f} |")
display(Markdown('\n'.join(rows)))